# 07 — 증강 가중치 XGBoost E (제출 2번 구성요소)

- 기존 B 모델/제출 1번은 수정하지 않습니다.
- 원본 변이 문자열에서 **각 토큰을 독립적으로** p=0.5/0.7/0.9 확률로 보존한 복사본 3개를 만듭니다. 공백 구분을 사용합니다.
- **Fold 학습 데이터만** 증강합니다. 검증/홀드아웃/Test는 원본 유지. 모든 E 피처는 증강 이후 재계산합니다.
- 가중치는 각 Fold의 원본 학습 라벨 빈도에서 계산하고 복사본에도 동일하게 적용합니다(평균 1 정규화).
- `src/`는 수정하지 않습니다. 실험용 CV는 증강 학습이 필요해 별도 루프로 계산하되 공통 `get_folds`, `macro_f1`, `log_result`, `save_predictions`를 사용합니다.
- 저장은 기본 OFF입니다. 모델 실행 결과를 확인한 뒤 최종 예측 저장 스위치를 켜세요.


**장치 정책:** 개발 CV와 선택적 홀드아웃은 GPU(cuda), 최종 전체 Train OOF와 Test 예측은 CPU로 생성합니다. 두 장치의 결과가 동일하다고 가정하지 않습니다.

In [1]:
from pathlib import Path
import sys, json, re, warnings
import numpy as np
import pandas as pd
from sklearn.base import clone
from xgboost import XGBClassifier
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p/'src'/'config.py').is_file()), None)
if ROOT is None: raise FileNotFoundError('프로젝트 폴더에서 실행하세요')
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from src.config import DATA_DIR, SEED, set_seed
from src.data_loader import load_data, split_xy, encode_target
from src.preprocessing import binarize_mutations
from src.features import mutation_count, mutation_type_counts, amino_acid_pair_counts, gene_truncation_flags
from src.validation import get_folds
from src.evaluation import macro_f1, log_result, save_predictions
from scripts.fixed_holdout_manifest import verify
set_seed(SEED)
EXPERIMENT_DEVICE = 'cuda'  # 개발 CV / 선택적 홀드아웃: RTX 5070
FINAL_DEVICE = 'cpu'      # 팀 규정: 최종 전체 OOF / Test: CPU
RUN_FULL_CV = True  # 개발 CV와 검증 완료 후 True
SAVE_ARTIFACTS = True  # 전체 OOF/Test 생성 후 True
print('ROOT:', ROOT, 'seed:', SEED)
print('실험 장치:', EXPERIMENT_DEVICE, '| 최종 제출 후보 장치:', FINAL_DEVICE)
# XGBoost GPU 학습에는 PyTorch가 필요하지 않습니다.
# 실제 CUDA 사용 가능 여부는 첫 XGBoost fit에서 확인됩니다.
import xgboost as xgb
print('XGBoost 버전:', xgb.__version__)
if EXPERIMENT_DEVICE == 'cuda':
    print('GPU 실험 요청: XGBoost device=cuda (실제 사용 여부는 학습 로그 확인)')


ROOT: C:\Users\user\Desktop\AI_Projects\ah07-cancer-subtype seed: 42
실험 장치: cuda | 최종 제출 후보 장치: cpu
XGBoost 버전: 3.4.1
GPU 실험 요청: XGBoost device=cuda (실제 사용 여부는 학습 로그 확인)


## 1. 데이터·고정 홀드아웃·E 피처

In [2]:
train, test = load_data(DATA_DIR)
X_raw, y_raw = split_xy(train)
y, le = encode_target(y_raw)
X_test_raw = test[X_raw.columns].copy()
assert X_raw.columns.equals(X_test_raw.columns)
assert len(le.classes_) == 26
K = len(le.classes_)
dev_idx, hold_idx = verify()  # 생성 금지
assert not np.intersect1d(dev_idx, hold_idx).size

def make_E(X):
    parts = [binarize_mutations(X), np.log1p(mutation_count(X)),
             np.log1p(mutation_type_counts(X)), np.log1p(amino_acid_pair_counts(X)),
             gene_truncation_flags(X)]
    out = pd.concat(parts, axis=1).astype(np.float32)
    assert out.index.equals(X.index) and not out.columns.has_duplicates
    return out

X_dev_raw = X_raw.iloc[dev_idx].reset_index(drop=True)
y_dev = np.asarray(y)[dev_idx]
folds_dev = get_folds(y_dev, X=X_dev_raw, classes=le.classes_)
print('개발셋:', len(dev_idx), '고정 홀드아웃:', len(hold_idx))

VERIFIED: dev=4960, holdout=1241; same IDs and row signatures
개발셋: 4960 고정 홀드아웃: 1241


## 2. 변이 토큰 독립 삭제 증강 — 자체 단위 검사 포함

In [3]:
def augment_raw(X, keep_probability, seed):
    """각 비-WT 셀의 공백 구분 토큰을 독립적으로 남김. 모두 삭제되면 WT."""
    assert 0 <= keep_probability <= 1
    rng = np.random.default_rng(seed)
    arr = X.to_numpy(dtype=object, copy=True)
    # 전체 4,384개 컬럼을 Python 이중 루프로 돌지 않고 변이가 있는 셀만 처리
    active = pd.notna(arr) & (arr != 'WT') & (arr != '')
    for i, j in np.argwhere(active):
        tokens = str(arr[i, j]).split()
        if not tokens:
            continue
        mask = rng.random(len(tokens)) < keep_probability
        arr[i, j] = ' '.join(t for t, keep in zip(tokens, mask) if keep) or 'WT'
    return pd.DataFrame(arr, index=X.index, columns=X.columns)

# 데이터 무관한 간단한 기능 검사
_demo = pd.DataFrame({'G1':['R273C R248Q','WT'], 'G2':['V600E','WT']})
assert augment_raw(_demo, 1.0, 42).equals(_demo)
assert augment_raw(_demo, 0.0, 42).eq('WT').all().all()
assert augment_raw(_demo, 0.7, 42).equals(augment_raw(_demo, 0.7, 42))
print('증강 토큰 분리·삭제·재현성 테스트 통과')

증강 토큰 분리·삭제·재현성 테스트 통과


## 3. 가중치 및 모델 설정 — 기존 B와 동일

In [4]:
PARAMS = dict(n_estimators=250, max_depth=3, learning_rate=0.05,
              subsample=0.8, colsample_bytree=0.8,
              objective='multi:softprob', num_class=K, eval_metric='mlogloss',
              tree_method='hist', random_state=SEED, n_jobs=-1)

def build_model(device):
    return XGBClassifier(**PARAMS, device=device)
KEEP_PROBS = (0.5, 0.7, 0.9)

def sample_weights_from_original(y_original):
    counts = np.bincount(np.asarray(y_original, dtype=int), minlength=K)
    assert np.all(counts > 0), '학습 fold에 누락된 클래스가 있습니다'
    w = 1 / np.sqrt(counts[np.asarray(y_original, dtype=int)])
    return (w / w.mean()).astype(np.float32)

def augmented_training_data(raw_train, y_train, fold_seed):
    raw_train = raw_train.reset_index(drop=True)
    y_train = np.asarray(y_train, dtype=int)
    blocks = [raw_train]
    for copy_i, p in enumerate(KEEP_PROBS):
        blocks.append(augment_raw(raw_train, p, fold_seed + copy_i * 1009 + 1))
    augmented = pd.concat(blocks, ignore_index=True)
    y_aug = np.tile(y_train, len(blocks))
    w_aug = np.tile(sample_weights_from_original(y_train), len(blocks))
    E_aug = make_E(augmented)
    assert len(E_aug) == len(y_aug) == len(w_aug) == 4 * len(raw_train)
    return E_aug, y_aug, w_aug

def check_proba(p, n):
    p = np.asarray(p, dtype=np.float32)
    assert p.shape == (n, K) and np.isfinite(p).all()
    assert np.allclose(p.sum(axis=1), 1, atol=1e-4)
    return p

## 4. 개발셋 Group CV — 검증 Fold 증강 금지

In [5]:
def augmented_cv(raw, labels, folds, tag, device):
    labels = np.asarray(labels, dtype=int)
    oof = np.full((len(labels), K), np.nan, dtype=np.float32)
    seen = np.zeros(len(labels), dtype=np.int8)
    scores = []
    for i, (tr, va) in enumerate(folds):
        assert len(np.intersect1d(tr, va)) == 0
        assert not seen[va].any()
        E_tr, y_tr, w_tr = augmented_training_data(raw.iloc[tr], labels[tr], SEED + i * 10000)
        E_va = make_E(raw.iloc[va].reset_index(drop=True))  # 원본 검증만
        assert E_tr.columns.equals(E_va.columns)
        m = build_model(device)
        m.fit(E_tr, y_tr, sample_weight=w_tr)
        assert np.array_equal(m.classes_, np.arange(K))
        oof[va] = check_proba(m.predict_proba(E_va), len(va))
        seen[va] = 1
        score = macro_f1(labels[va], oof[va].argmax(axis=1))
        scores.append(float(score))
        print(f'{tag} fold {i+1}: Macro F1 {score:.4f}')
    assert np.all(seen == 1) and np.isfinite(oof).all()
    result = {'scores': scores, 'mean': float(np.mean(scores)),
              'std': float(np.std(scores)), 'oof': oof.argmax(axis=1), 'oof_proba': oof}
    print(f'{tag} CV mean={result["mean"]:.4f} std={result["std"]:.4f}; OOF F1={macro_f1(labels, result["oof"]):.4f}')
    return result

cv_dev_aug = augmented_cv(X_dev_raw, y_dev, folds_dev, '개발 GPU', EXPERIMENT_DEVICE)

c:\Users\user\Desktop\AI_Projects\ah07-cancer-subtype\.venv\Lib\site-packages\xgboost\core.py:569: UserWarning: [13:51:46] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


개발 GPU fold 1: Macro F1 0.4776
개발 GPU fold 2: Macro F1 0.4533
개발 GPU fold 3: Macro F1 0.4782
개발 GPU fold 4: Macro F1 0.4594
개발 GPU fold 5: Macro F1 0.4762
개발 GPU CV mean=0.4689 std=0.0105; OOF F1=0.4702


## 5. 고정 홀드아웃 (선택적 1회 검증)

홀드아웃을 반복적으로 보면서 파라미터를 조정하지 마세요. 개발 CV를 먼저 검토한 뒤 팀 규칙에 따라 한 번만 실행합니다.

In [6]:
RUN_HOLDOUT = False
if RUN_HOLDOUT:
    E_tr, y_tr, w_tr = augmented_training_data(X_raw.iloc[dev_idx], y[dev_idx], SEED + 70000)
    E_hold = make_E(X_raw.iloc[hold_idx].reset_index(drop=True))
    assert E_tr.columns.equals(E_hold.columns)
    hold_model = build_model(EXPERIMENT_DEVICE).fit(E_tr, y_tr, sample_weight=w_tr)
    p_hold = check_proba(hold_model.predict_proba(E_hold), len(hold_idx))
    print('증강 XGB 홀드아웃 Macro F1:', macro_f1(y[hold_idx], p_hold.argmax(axis=1)))
else:
    print('홀드아웃 미실행 (기본 설정)')

홀드아웃 미실행 (기본 설정)


## 6. 전체 Train OOF 및 Test 예측

유정님과 앙상블하려면 **전체 Train 원본 순서**의 OOF와 **Test 원본 순서**의 확률이 필요합니다. `RUN_FULL_CV=True`로 바꾸면 전체 Group CV를 수행합니다. 이때 전체 Train CV는 홀드아웃을 포함하므로 별도의 독립 홀드아웃 검증으로 해석하지 않습니다.

**주의:** `RUN_FULL_CV=True`를 켜면 전체 OOF 5-Fold와 최종 Test 모델이 **CPU로 학습**됩니다. GPU 개발 CV 점수를 최종 CPU CV 점수로 대체하지 마세요.

In [7]:
if RUN_FULL_CV:
    folds_full = get_folds(y, X=X_raw, classes=le.classes_)
    cv_full_aug = augmented_cv(X_raw, y, folds_full, '전체 Train CPU', FINAL_DEVICE)
    E_full, y_full, w_full = augmented_training_data(X_raw, y, SEED + 90000)
    E_test = make_E(X_test_raw.reset_index(drop=True))
    assert E_full.columns.equals(E_test.columns)
    final_model = build_model(FINAL_DEVICE).fit(E_full, y_full, sample_weight=w_full)
    test_proba = check_proba(final_model.predict_proba(E_test), len(test))
    print('OOF:', cv_full_aug['oof_proba'].shape, 'Test:', test_proba.shape)
else:
    print('전체 OOF/Test 학습은 RUN_FULL_CV=True로 켜세요')

전체 Train CPU fold 1: Macro F1 0.4676
전체 Train CPU fold 2: Macro F1 0.5038
전체 Train CPU fold 3: Macro F1 0.4744
전체 Train CPU fold 4: Macro F1 0.4923
전체 Train CPU fold 5: Macro F1 0.4808
전체 Train CPU CV mean=0.4838 std=0.0129; OOF F1=0.4850
OOF: (6201, 26) Test: (2546, 26)


## 7. 팀 표준 파일 저장 — 결과 확인 후 실행

In [8]:
NAME = 'xgb_E_depth3_invsqrt_mutdrop_050709_group_full_1110'
if SAVE_ARTIFACTS:
    assert RUN_FULL_CV and 'cv_full_aug' in globals() and 'test_proba' in globals()
    log_result(owner='A_model', notebook='07_xgb_weighted_mutation_augmentation.ipynb',
               experiment=NAME, features='E_9174', model='XGBClassifier',
               params={**PARAMS, 'augmentation_keep_probs': KEEP_PROBS, 'copies': 3,
                       'sample_weight': 'inverse_sqrt_original_fold_counts_mean1',
                       'development_device': EXPERIMENT_DEVICE, 'final_device': FINAL_DEVICE},
               cv=cv_full_aug, notes='train-fold-only mutation token dropout; full train OOF; no test augmentation')
    save_predictions(owner='A_model', name=NAME,
                     oof_proba=cv_full_aug['oof_proba'], test_proba=test_proba,
                     classes=le.classes_, cv=cv_full_aug,
                     notes='XGB weighted + 3 mutation-dropout copies; train-only; classes in le order')
else:
    print('저장 비활성화: 결과 검토 후 SAVE_ARTIFACTS=True로 변경')

기록: log_A_model.csv ← xgb_E_depth3_invsqrt_mutdrop_050709_group_full_1110 (CV 0.4838)
예측 저장: A_model__xgb_E_depth3_invsqrt_mutdrop_050709_group_full_1110 (CV 0.4838) → experiments/preds/


## 8. 유정님 공유 / PR 체크

- `experiments/preds/A_model__xgb_E_depth3_invsqrt_mutdrop_050709_group_full_1110__oof.npy`
- `experiments/preds/A_model__xgb_E_depth3_invsqrt_mutdrop_050709_group_full_1110__test.npy`
- `experiments/preds/A_model__xgb_E_depth3_invsqrt_mutdrop_050709_group_full_1110__meta.json`

**유정님:** 증강 LR OOF/Test와 동일한 행·클래스 순서 확인 후 확률을 `(XGB + LR)/2`로 결합. 제출 CSV는 유정님 담당. 제출 1번 LB 대비 팀장님 기준으로 판단합니다.

주의: 이 노트북은 전체 실데이터 학습을 여기서 수행하지 않았으므로 점수는 실행 후 확인하세요. `src/` 변경 금지. 노트북 실행 출력에 환자 식별정보 포함 금지.

- GPU 개발 CV는 실험용이며, 앙상블에 전달하는 OOF/Test `.npy`는 **모두 CPU 학습 결과**입니다.